# MP4 Converter

Bağımsız video dönüştürücü. Tüm videoları **H.264 mp4** formatına çevirir. Watermark pipeline'ından bağımsız çalışır — herhangi bir projede kullanılabilir.

---

## 📁 Klasör yapısı

Notebook ilk çalıştırıldığında Drive'da şu yapıyı **otomatik** oluşturur:

```
/content/drive/MyDrive/mp4_converter/
├── input/         ← videolarını buraya koy (.webm, .mp4, .avi, .mov, .mkv)
└── output/        ← dönüştürülmüş .mp4'ler buraya yazılır
```

| | Yol | Ne yapılır |
|---|---|---|
| **INPUT**  | `mp4_converter/input/`  | Sen videolarını buraya yüklersin. Dosyalar değiştirilmez. |
| **OUTPUT** | `mp4_converter/output/` | Notebook çıktıyı buraya yazar (aynı klasör yapısıyla). |

İç klasör yapısı korunur — `input/foo/bar.webm` → `output/foo/bar.mp4`

---

## 🔄 Akış

1. Drive mount
2. `mp4_converter/input/` ve `mp4_converter/output/` klasörlerini oluştur (yoksa)
3. `input/` altındaki tüm videoları tara
4. Her birini ffmpeg ile H.264 mp4'e çevir → `output/`'a yaz
5. Özet rapor

**Idempotent:** Tekrar çalıştırırsan zaten dönüştürülmüş dosyaları atlar.

---

## 🔌 Başka pipeline ile kullanım

Watermark detection (veya başka bir notebook) bu çıktıyı tüketmek istiyorsa:

```python
INPUT_FOLDER = "/content/drive/MyDrive/mp4_converter/output"
```

In [ ]:
# ============================================================
# 1 — Google Drive bağlantısı
# ============================================================
from google.colab import drive
import os

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
print("✅ Drive bağlandı.")

In [ ]:
# ============================================================
# 2 — AYARLAR  ← gerekirse buradan değiştir
# ============================================================
from pathlib import Path

# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# MP4 Converter'ın ana klasörü (Drive üzerinde)
# Bunun altında otomatik olarak input/ ve output/ oluşturulur.
CONVERTER_ROOT = Path("/content/drive/MyDrive/mp4_converter")
# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

INPUT_ROOT  = CONVERTER_ROOT / "input"
OUTPUT_ROOT = CONVERTER_ROOT / "output"

# Hangi video formatları taransın?
VIDEO_EXTENSIONS = (".mp4", ".avi", ".mov", ".mkv", ".webm")

# .mp4 dosyalar da yeniden encode edilsin mi?
#   False → .mp4'ler olduğu gibi kopyalanır (hızlı, varsayılan)
#   True  → .mp4'ler de H.264'e çevrilir (H.265/HEVC sorunlarını çözer, yavaş)
FORCE_REENCODE_MP4 = False

# ffmpeg kalite ayarları
CRF    = "23"      # 18-28 arası: küçük = daha kaliteli + büyük dosya
PRESET = "fast"    # ultrafast / superfast / veryfast / faster / fast / medium / slow

# Klasörleri oluştur
CONVERTER_ROOT.mkdir(parents=True, exist_ok=True)
INPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"📂 Converter root : {CONVERTER_ROOT}")
print(f"📥 INPUT  klasörü : {INPUT_ROOT}")
print(f"📤 OUTPUT klasörü : {OUTPUT_ROOT}")
print(f"🎞️  Aranan formatlar: {', '.join(VIDEO_EXTENSIONS)}")

# Kullanıcı uyarısı
n_in_input = sum(1 for _ in INPUT_ROOT.rglob("*") if _.is_file())
if n_in_input == 0:
    print("\n⚠️  INPUT klasörü boş — videolarını şuraya yükle:")
    print(f"   {INPUT_ROOT}")

In [ ]:
# ============================================================
# 3 — ffmpeg kontrolü (gerekirse kurar)
# ============================================================
import subprocess

r = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
if r.returncode != 0:
    print("ffmpeg bulunamadı, kuruluyor...")
    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)
    r = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)

print("✅", r.stdout.splitlines()[0])

In [ ]:
# ============================================================
# 4 — INPUT klasöründeki videoları tara
# ============================================================
videos = sorted([
    p for p in INPUT_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
])

print(f"🔍 {len(videos)} video bulundu.\n")

# Format dağılımı
from collections import Counter
ext_counts = Counter(v.suffix.lower() for v in videos)
if ext_counts:
    print("Format dağılımı:")
    for ext, n in ext_counts.most_common():
        print(f"  {ext:<8} → {n} dosya")

    print("\nİlk 10 dosya:")
    for v in videos[:10]:
        print(" -", v.relative_to(INPUT_ROOT))
    if len(videos) > 10:
        print(f" ... (+{len(videos)-10} tane daha)")
else:
    print(f"⚠️  INPUT klasörü boş: {INPUT_ROOT}")

In [ ]:
# ============================================================
# 5 — Dönüştürme fonksiyonu
# ============================================================
import shutil

def convert_video(src: Path, dst: Path, force_reencode_mp4: bool = False) -> dict:
    """Tek bir videoyu çıktı yoluna dönüştürür / kopyalar."""
    dst.parent.mkdir(parents=True, exist_ok=True)

    # Zaten dönüştürülmüş — atla
    if dst.exists() and dst.stat().st_size > 0:
        return {"status": "SKIP", "reason": "already exists"}

    # mp4 ve re-encode istenmiyorsa direkt kopyala (hızlı yol)
    if src.suffix.lower() == ".mp4" and not force_reencode_mp4:
        try:
            shutil.copy2(src, dst)
            return {"status": "COPY"}
        except Exception as e:
            return {"status": "ERROR", "stderr": str(e)}

    # Diğer tüm formatlar → ffmpeg ile H.264 mp4'e çevir
    cmd = [
        "ffmpeg", "-y", "-i", str(src),
        "-c:v", "libx264", "-preset", PRESET, "-crf", CRF,
        "-pix_fmt", "yuv420p",
        "-c:a", "aac", "-b:a", "128k",
        "-movflags", "+faststart",
        str(dst),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        if dst.exists():
            dst.unlink()  # bozuk yarım dosyayı temizle
        return {"status": "ERROR", "stderr": result.stderr[-400:]}
    return {"status": "CONVERTED"}

print("✅ Fonksiyon hazır.")

In [ ]:
# ============================================================
# 6 — Tüm videoları dönüştür
# ============================================================
from tqdm.auto import tqdm

results = {}
for src in tqdm(videos, desc="Dönüştürme"):
    rel = src.relative_to(INPUT_ROOT)
    dst = OUTPUT_ROOT / rel.with_suffix(".mp4")
    res = convert_video(src, dst, FORCE_REENCODE_MP4)
    results[str(rel)] = res
    print(f"  [{res['status']:<9}] {rel}")

print("\n🏁 Bitti.")

In [ ]:
# ============================================================
# 7 — Özet
# ============================================================
from collections import Counter

counts = Counter(r["status"] for r in results.values())
print("=" * 40)
print("ÖZET")
print("=" * 40)
for status, n in counts.most_common():
    icon = {"CONVERTED": "🔄", "COPY": "📋", "SKIP": "⏭️", "ERROR": "❌"}.get(status, "•")
    print(f"  {icon} {status:<10} : {n}")

errors = {k: v for k, v in results.items() if v["status"] == "ERROR"}
if errors:
    print(f"\n⚠️  {len(errors)} dosya dönüştürülemedi:")
    for k, v in errors.items():
        print(f"\n  - {k}")
        print(f"    {v.get('stderr','')[:300]}")
else:
    print("\n✅ Tüm videolar başarıyla işlendi.")

print(f"\n📤 Çıktı klasörü: {OUTPUT_ROOT}")
print("\n👉 Başka pipeline'ın bunu tüketmesini istiyorsan input path'ini şu yap:")
print(f'   "{OUTPUT_ROOT}"')